# Proximal Policy Optimization (PPO) from Scratch: LunarLander-v3

This notebook implements PPO completely from scratch using **PyTorch** and **Gymnasium**.

**What you will build, step by step:**
1. LunarLander-v3 environment walkthrough
2. Actor-Critic network (shared backbone, two heads)
3. GAE (Generalized Advantage Estimation) — the advantage calculator
4. PPO Clipped Objective — the core innovation
5. Rollout Buffer — stores one batch of experience per iteration
6. Full PPO training loop (collect → compute advantages → K-epoch update)
7. Training curves, policy diagnostics, and landing visualizations
8. Ablation: clipping vs no clipping (direct demonstration of why PPO works)


## Cell 1 — Imports

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque
import warnings, random, time
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
print(f'Gym     : {gym.__version__}')


## Cell 2 — LunarLander-v3 Environment Walkthrough

LunarLander is substantially harder than CartPole:
- **8-dimensional state** vs 4 for CartPole
- **4 discrete actions** vs 2
- **Dense reward shaping** with multiple components
- **Delayed landing reward** — the agent must survive ~hundreds of steps before getting the big reward
- **Solved threshold**: mean reward ≥ 200 over 100 consecutive episodes


In [ ]:
env = gym.make('LunarLander-v3')
env.reset(seed=SEED)

print('=== LunarLander-v3 Environment ===')
print(f'Observation space : {env.observation_space}')
print(f'  Shape           : {env.observation_space.shape}')
print(f'Action space      : {env.action_space}')
print(f'  n_actions       : {env.action_space.n}')
print()

obs, _ = env.reset(seed=SEED)
labels = ['x position','y position','x velocity','y velocity',
          'angle','angular velocity','left leg contact','right leg contact']
print('Initial state:')
for i, (lb, v) in enumerate(zip(labels, obs)):
    print(f'  [{i}] {lb:22s}: {v:8.4f}')
print()
print('Actions:')
for a, desc in enumerate(['Do nothing','Fire left engine','Fire main engine','Fire right engine']):
    print(f'  [{a}] {desc}')
print()
print('Reward components:')
print('  +100 to +140 : landing on pad (more for centered, slow landing)')
print('  -100         : crash')
print('  +10          : each leg making ground contact')
print('  -0.3/frame   : main engine firing (fuel cost)')
print('  -0.03/frame  : side engine firing (fuel cost)')
print('  Solved when  : mean reward >= 200 over 100 consecutive episodes')
print()

# Quick random episode to show reward structure
state, _ = env.reset(seed=SEED)
ep_reward = 0
rewards_by_step = []
for step in range(200):
    action = env.action_space.sample()
    state, reward, terminated, truncated, _ = env.step(action)
    ep_reward += reward
    rewards_by_step.append(reward)
    if terminated or truncated:
        break

print(f'Random episode: {step+1} steps, total reward = {ep_reward:.1f}')
print(f'  Min reward per step: {min(rewards_by_step):.3f}')
print(f'  Max reward per step: {max(rewards_by_step):.3f}')
print(f'  Mean reward/step   : {np.mean(rewards_by_step):.3f}')
env.close()


## Cell 3 — Actor-Critic Network

PPO uses a **shared backbone** with two output heads:
- **Actor head**: outputs action logits → probabilities via Categorical
- **Critic head**: outputs a single scalar V(s)

Sharing the backbone means the feature representation learned for value estimation also benefits the policy, and vice versa. This is more parameter-efficient and often trains more stably than separate networks.

**Key design choices:**
- `tanh` activations (instead of ReLU) — common for continuous-state RL; avoids dead neurons
- Orthogonal weight initialization — shown empirically to improve training stability in PPO
- Small weight scale on output layers — keeps initial policy near-uniform and V(s) near zero


In [ ]:
class ActorCritic(nn.Module):
    """
    Shared-backbone Actor-Critic for PPO.
    
    Architecture:
        state(8) → [Linear(64) → tanh → Linear(64) → tanh]  ← shared trunk
                    ↓                                  ↓
              actor_head                         critic_head
              Linear(64→4)                       Linear(64→1)
              → action logits                    → V(s) scalar
    """
    
    def __init__(self, n_states, n_actions, hidden_size=64):
        super().__init__()
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(n_states, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        
        # Actor head: outputs logits for each action
        self.actor_head = nn.Linear(hidden_size, n_actions)
        
        # Critic head: outputs scalar state value
        self.critic_head = nn.Linear(hidden_size, 1)
        
        # Orthogonal initialization — empirically good for PPO
        self._init_weights()
    
    def _init_weights(self):
        for module in self.shared:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.zeros_(module.bias)
        # Smaller gain for output heads — keeps initial policy near-uniform
        nn.init.orthogonal_(self.actor_head.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic_head.weight, gain=1.0)
        nn.init.zeros_(self.actor_head.bias)
        nn.init.zeros_(self.critic_head.bias)
    
    def forward(self, state):
        """Returns (action_logits, state_value)."""
        features = self.shared(state)
        logits   = self.actor_head(features)
        value    = self.critic_head(features).squeeze(-1)
        return logits, value
    
    def get_action(self, state):
        """
        Sample action from π(·|state).
        Returns: action, log_prob, value estimate
        """
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            logits, value = self.forward(state_t)
        dist     = Categorical(logits=logits)
        action   = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob.item(), value.item()
    
    def evaluate_actions(self, states, actions):
        """
        Evaluate a batch of (state, action) pairs.
        Used during the PPO update phase.
        Returns: log_probs, values, entropy
        """
        logits, values = self.forward(states)
        dist      = Categorical(logits=logits)
        log_probs = dist.log_prob(actions)
        entropy   = dist.entropy()
        return log_probs, values, entropy
    
    def get_value(self, state):
        """Critic only — for GAE computation."""
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            _, value = self.forward(state_t)
        return value.item()


# ── Inspect ──────────────────────────────────────────────────────────────────
n_states  = 8
n_actions = 4

ac_net = ActorCritic(n_states, n_actions).to(device)
total_p = sum(p.numel() for p in ac_net.parameters())
print('Actor-Critic Network:')
print(ac_net)
print(f'\nTotal parameters: {total_p:,}')
print()

# Demo forward pass
state_demo = np.zeros(8, dtype=np.float32)
state_demo[1] = 1.5   # y position = 1.5 (hovering)
action, log_prob, value = ac_net.get_action(state_demo)
print(f'Demo state: y=1.5 (hovering), all else zero')
print(f'  Sampled action : {action} ({["nothing","L-engine","main","R-engine"][action]})')
print(f'  log π(action)  : {log_prob:.4f}')
print(f'  V(state)       : {value:.4f}  (near 0 at init — expected)')
print()

# Check initial policy is near-uniform (due to small weight init)
s_t = torch.tensor(state_demo, dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    logits, _ = ac_net.forward(s_t)
    probs = torch.softmax(logits, dim=-1)
print(f'Initial action probabilities: {probs.cpu().numpy().round(4)}')
print('(Should be near-uniform [0.25, 0.25, 0.25, 0.25] due to small output weight init)')


## Cell 4 — Generalized Advantage Estimation (GAE)

GAE is the advantage estimator used by PPO. It produces a weighted blend of
TD residuals at different time horizons, controlled by `λ`:

```
δ_t   = r_t + γ·V(s_{t+1}) − V(s_t)      ← TD residual
A_t   = δ_t + (γλ)·δ_{t+1} + (γλ)²·δ_{t+2} + ...
```

Computed efficiently by a single backward pass through the stored TD residuals.

**The return target** for the critic is `G_t = A_t + V(s_t)`, which equals the
discounted-return estimate used to train V(s).


In [ ]:
def compute_gae(rewards, values, dones, last_value, gamma=0.99, gae_lambda=0.95):
    """
    Compute GAE advantages and return targets for a collected rollout.
    
    Parameters
    ----------
    rewards    : list[float]  length T
    values     : list[float]  length T  (V(s_t) from critic)
    dones      : list[bool]   length T  (True if episode ended at step t)
    last_value : float        V(s_{T+1}) — value of state AFTER rollout ends
    gamma      : float        discount factor
    gae_lambda : float        GAE smoothing (0=TD, 1=Monte Carlo)
    
    Returns
    -------
    advantages : np.ndarray  shape (T,)
    returns    : np.ndarray  shape (T,)  — critic training targets
    """
    T          = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    
    gae = 0.0
    for t in reversed(range(T)):
        # Bootstrap value at t+1 (0 if episode ended at step t)
        next_value = last_value if t == T-1 else values[t+1]
        next_value = next_value * (1.0 - float(dones[t]))
        
        # TD residual: how much better/worse was this step than expected?
        delta = rewards[t] + gamma * next_value - values[t]
        
        # Accumulate GAE (exponentially weighted sum of deltas)
        gae   = delta + gamma * gae_lambda * (1.0 - float(dones[t])) * gae
        advantages[t] = gae
    
    # Return targets = advantage + baseline (value function)
    returns = advantages + np.array(values, dtype=np.float32)
    
    return advantages, returns


# ── Visualize GAE on a synthetic rollout ────────────────────────────────────
np.random.seed(SEED)
T_demo = 50

# Synthetic rollout: small rewards, then big landing reward at the end
rewards_demo = [-0.1] * 49 + [100.0]
values_demo  = list(np.linspace(0, 80, 50))      # improving value estimates
dones_demo   = [False] * 49 + [True]
last_val     = 0.0

adv_lambda_0,  ret_0  = compute_gae(rewards_demo, values_demo, dones_demo, last_val, gae_lambda=0.0)
adv_lambda_95, ret_95 = compute_gae(rewards_demo, values_demo, dones_demo, last_val, gae_lambda=0.95)
adv_lambda_1,  ret_1  = compute_gae(rewards_demo, values_demo, dones_demo, last_val, gae_lambda=1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('GAE: Effect of λ on Advantage Estimates (Synthetic Rollout)', fontsize=12)

t = np.arange(T_demo)
axes[0].plot(t, adv_lambda_0,   color='red',   linewidth=2, label='λ=0.00 (TD only, low var, biased)')
axes[0].plot(t, adv_lambda_95,  color='green', linewidth=2, label='λ=0.95 (PPO default)')
axes[0].plot(t, adv_lambda_1,   color='blue',  linewidth=2, label='λ=1.00 (MC, high var, unbiased)')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(49, color='orange', linestyle=':', label='Landing reward')
axes[0].set_title('Advantage A_t at each step'); axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Advantage'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(t, ret_0,   color='red',   linewidth=2, label='λ=0.00')
axes[1].plot(t, ret_95,  color='green', linewidth=2, label='λ=0.95')
axes[1].plot(t, ret_1,   color='blue',  linewidth=2, label='λ=1.00')
axes[1].set_title('Return targets G_t = A_t + V(s_t)'); axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Return target'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Key insight:')
print('  λ=0 (TD): advantage is local, noisy — only looks 1 step ahead')
print('  λ=1 (MC): advantage propagates far back — early steps "credited" for landing')
print('  λ=0.95  : blends both — some credit propagation, controlled variance')


## Cell 5 — Rollout Buffer

Unlike DQN's replay buffer (random sample from large history), PPO uses a
**rollout buffer**: a fixed-size batch that is entirely collected, then entirely consumed,
then discarded. No random sampling across past iterations — only the current batch.

The buffer stores:
- `states`, `actions`, `log_probs_old`, `values` — collected during rollout
- After rollout: `advantages`, `returns` — computed via GAE

During the update phase, it yields random **mini-batches** from within the current rollout.


In [ ]:
class RolloutBuffer:
    """
    Fixed-size buffer that stores one PPO rollout (N_STEPS transitions).
    
    Two-phase usage:
      Phase 1 (collect): env.step() → buffer.add(...)
      Phase 2 (update):  buffer.compute_advantages() → buffer.get_batches()
    """
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        """Clear all stored data."""
        self.states       = []
        self.actions      = []
        self.log_probs    = []   # log π_old(a_t|s_t) — stored at collection time
        self.rewards      = []
        self.values       = []   # V(s_t) — stored at collection time
        self.dones        = []
        # Computed after rollout:
        self.advantages   = None
        self.returns      = None
    
    def add(self, state, action, log_prob, reward, value, done):
        """Add one transition to the buffer."""
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)
    
    def compute_advantages(self, last_value, gamma, gae_lambda):
        """
        Compute GAE advantages and return targets.
        Call this once after collecting the full rollout.
        """
        self.advantages, self.returns = compute_gae(
            self.rewards, self.values, self.dones,
            last_value, gamma, gae_lambda
        )
        # Normalize advantages — zero mean, unit std within the batch
        # Same variance reduction as return normalization in REINFORCE
        self.advantages = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)
    
    def get_batches(self, batch_size):
        """
        Yield random mini-batches from the current rollout.
        Used during the K-epoch update loop.
        """
        n = len(self.states)
        indices = np.random.permutation(n)
        
        # Convert lists to tensors once
        states_t     = torch.tensor(np.array(self.states),    dtype=torch.float32).to(device)
        actions_t    = torch.tensor(np.array(self.actions),   dtype=torch.long   ).to(device)
        log_probs_t  = torch.tensor(np.array(self.log_probs), dtype=torch.float32).to(device)
        advantages_t = torch.tensor(self.advantages,          dtype=torch.float32).to(device)
        returns_t    = torch.tensor(self.returns,             dtype=torch.float32).to(device)
        
        for start in range(0, n, batch_size):
            batch_idx = indices[start : start + batch_size]
            yield (
                states_t[batch_idx],
                actions_t[batch_idx],
                log_probs_t[batch_idx],
                advantages_t[batch_idx],
                returns_t[batch_idx],
            )
    
    def __len__(self):
        return len(self.states)


# Quick test
buf = RolloutBuffer()
for _ in range(10):
    buf.add(np.zeros(8), 0, -1.4, 1.0, 0.5, False)
buf.compute_advantages(last_value=0.0, gamma=0.99, gae_lambda=0.95)
print(f'Buffer size: {len(buf)}')
print(f'Advantages shape: {buf.advantages.shape}')
print(f'Returns    shape: {buf.returns.shape}')
print(f'Advantages (first 5): {buf.advantages[:5].round(4)}')
print()
batches = list(buf.get_batches(batch_size=4))
print(f'Number of mini-batches (size 4 from 10): {len(batches)}')
print(f'First batch states shape: {batches[0][0].shape}')


## Cell 6 — PPO Loss Function (Step-by-Step)

This cell implements and demystifies the PPO clipped objective.
We trace through a concrete numerical example before wiring it into the agent.

**The three components:**
```
L^PPO = L^CLIP − c₁·L^VF + c₂·H

L^CLIP = mean( min(r·A, clip(r, 1−ε, 1+ε)·A) )     ← policy loss
L^VF   = mean( (V(s) − G)² )                         ← value loss
H      = mean( entropy of π(·|s) )                   ← entropy bonus
```


In [ ]:
def ppo_loss(ac_net, states, actions, old_log_probs, advantages, returns,
             clip_epsilon=0.2, value_coef=0.5, entropy_coef=0.01):
    """
    Compute the PPO loss for one mini-batch.
    
    Parameters
    ----------
    ac_net        : ActorCritic network
    states        : (B, n_states)  tensor
    actions       : (B,)           tensor of ints
    old_log_probs : (B,)           tensor  — log π_old(a|s) stored during collection
    advantages    : (B,)           tensor  — GAE advantages (normalized)
    returns       : (B,)           tensor  — return targets for critic
    clip_epsilon  : float          PPO clipping parameter ε
    value_coef    : float          c₁, weight on value loss
    entropy_coef  : float          c₂, weight on entropy bonus
    
    Returns
    -------
    total_loss, policy_loss, value_loss, entropy, clip_fraction
    """
    # ── Evaluate current policy on stored (state, action) pairs ─────────────
    new_log_probs, values, entropy = ac_net.evaluate_actions(states, actions)
    
    # ── Policy loss: PPO clipped objective ───────────────────────────────────
    # Probability ratio r_t = π_new(a|s) / π_old(a|s)
    # In log space: log(r_t) = log π_new − log π_old
    # Then r_t = exp(log r_t)
    log_ratio = new_log_probs - old_log_probs
    ratio     = torch.exp(log_ratio)
    
    # Unclipped objective: r_t * A_t
    surr_unclipped = ratio * advantages
    
    # Clipped objective: clip(r_t, 1-ε, 1+ε) * A_t
    surr_clipped   = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
    
    # Take the pessimistic (min) of the two — this IS the PPO loss
    policy_loss    = -torch.min(surr_unclipped, surr_clipped).mean()
    
    # ── Value function loss ──────────────────────────────────────────────────
    value_loss     = F.mse_loss(values, returns)
    
    # ── Entropy bonus ────────────────────────────────────────────────────────
    entropy_mean   = entropy.mean()
    
    # ── Combined loss ────────────────────────────────────────────────────────
    # Negative sign: we maximize J (policy gradient) but minimize everything else
    total_loss     = policy_loss + value_coef * value_loss - entropy_coef * entropy_mean
    
    # ── Diagnostic: clip fraction ────────────────────────────────────────────
    # Fraction of samples where clipping was active
    # High clip_fraction → policy is changing too fast → may need smaller lr or ε
    clip_fraction  = ((ratio - 1.0).abs() > clip_epsilon).float().mean().item()
    
    return total_loss, policy_loss.item(), value_loss.item(), entropy_mean.item(), clip_fraction


# ── Numerical trace: understand the clipping ────────────────────────────────
print('=== PPO CLIPPING NUMERICAL TRACE ===')
print()

clip_eps = 0.2
cases = [
    ('Strong +advantage, large ratio', 0.5,  2.5),  # ratio outside clip, A > 0
    ('Strong +advantage, small ratio', 0.5,  1.1),  # ratio inside clip, A > 0
    ('Strong −advantage, large ratio', -0.5, 2.5),  # ratio outside clip, A < 0
    ('Strong −advantage, small ratio', -0.5, 0.8),  # ratio inside clip, A < 0
    ('Zero advantage', 0.0,  1.5),                  # A=0 → no gradient anyway
    ('Ratio=1 (no change)', 0.3, 1.0),              # perfectly on-policy
]

print(f'{"Case":42s}  {"A_t":>6}  {"r_t":>6}  {"Unclipped":>10}  {"Clipped":>8}  {"PPO(min)":>10}  {"Active?":>8}')
print('-' * 100)
for desc, A, r in cases:
    unclip  = r * A
    r_clip  = np.clip(r, 1-clip_eps, 1+clip_eps)
    clipped = r_clip * A
    ppo_val = min(unclip, clipped)
    active  = '✓ CLIPPED' if r_clip != r else '  (free)'
    print(f'{desc:42s}  {A:>6.2f}  {r:>6.2f}  {unclip:>10.4f}  {clipped:>8.4f}  {ppo_val:>10.4f}  {active:>8}')

print()
print('Key observations:')
print('  Row 1: r=2.5 > 1.2 with A>0 → clipped at 1.2 → gradient stops here')
print('  Row 3: r=2.5 > 1.2 with A<0 → unclipped wins (more negative) → still penalized')
print('         (the min picks the more conservative/pessimistic value)')
print('  Row 5: A=0 → PPO value=0 regardless → no gradient signal, as expected')

# ── Visualize clipped objective ──────────────────────────────────────────────
ratios  = np.linspace(0.0, 3.0, 300)
clip_e  = 0.2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PPO Clipped Objective: Gradient Behaviour', fontsize=13, fontweight='bold')

for ax, A, title in [(axes[0], 1.0, 'A_t = +1.0 (good action)'),
                      (axes[1], -1.0, 'A_t = −1.0 (bad action)')]:
    unclip  = ratios * A
    clip_r  = np.clip(ratios, 1-clip_e, 1+clip_e)
    clipped = clip_r * A
    ppo_obj = np.minimum(unclip, clipped)
    
    ax.plot(ratios, unclip,  'b--',  linewidth=2,   label='Unclipped r·A')
    ax.plot(ratios, clipped, 'r--',  linewidth=2,   label='Clipped r·A')
    ax.plot(ratios, ppo_obj, 'g-',   linewidth=3,   label='PPO min (actual)')
    ax.axvline(1.0,        color='gray', linestyle=':',  alpha=0.7, label='r=1 (no change)')
    ax.axvline(1+clip_e,   color='red',  linestyle='-.', alpha=0.5, label=f'1+ε={1+clip_e}')
    ax.axvline(1-clip_e,   color='red',  linestyle='-.', alpha=0.5, label=f'1-ε={1-clip_e}')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.fill_betweenx([-2,2], 1-clip_e, 1+clip_e, alpha=0.1, color='green', label='Free zone')
    ax.set_title(title); ax.set_xlabel('Probability ratio r_t = π_new/π_old')
    ax.set_ylabel('Objective value'); ax.set_ylim(-2, 2); ax.set_xlim(0, 3)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Cell 7 — PPO Agent

Puts everything together: Actor-Critic + Rollout Buffer + PPO update.

The two phases are clearly separated:
- **`collect_rollout()`**: runs the environment for `n_steps`, storing all transitions
- **`update()`**: runs K epochs of gradient updates over the collected batch


In [ ]:
class PPOAgent:
    """
    Proximal Policy Optimization (PPO-Clip) agent.
    
    Hyperparameters
    ---------------
    n_steps         : int    Steps collected per iteration (rollout length)
    k_epochs        : int    Gradient update epochs per iteration
    clip_epsilon    : float  PPO clipping parameter ε
    gamma           : float  Discount factor
    gae_lambda      : float  GAE smoothing parameter λ
    lr_actor        : float  Actor (+ shared trunk) learning rate
    lr_critic       : float  Critic head learning rate (can differ)
    value_coef      : float  c₁ — weight on value loss
    entropy_coef    : float  c₂ — weight on entropy bonus
    mini_batch_size : int    Mini-batch size within each epoch
    """
    
    def __init__(
        self,
        n_states,
        n_actions,
        n_steps         = 2048,
        k_epochs        = 10,
        clip_epsilon    = 0.2,
        gamma           = 0.99,
        gae_lambda      = 0.95,
        lr              = 3e-4,
        value_coef      = 0.5,
        entropy_coef    = 0.01,
        mini_batch_size = 64,
        max_grad_norm   = 0.5,
    ):
        self.n_steps         = n_steps
        self.k_epochs        = k_epochs
        self.clip_epsilon    = clip_epsilon
        self.gamma           = gamma
        self.gae_lambda      = gae_lambda
        self.value_coef      = value_coef
        self.entropy_coef    = entropy_coef
        self.mini_batch_size = mini_batch_size
        self.max_grad_norm   = max_grad_norm
        
        # Single Actor-Critic network
        self.ac_net    = ActorCritic(n_states, n_actions).to(device)
        self.optimizer = optim.Adam(self.ac_net.parameters(), lr=lr, eps=1e-5)
        
        # Rollout buffer — reused each iteration
        self.buffer = RolloutBuffer()
        
        # Diagnostics
        self.policy_losses  = []
        self.value_losses   = []
        self.entropies      = []
        self.clip_fractions = []
        self.approx_kls     = []
    
    # ────────────────────────────────────────────────────────────────────────
    # Phase 1: Collect rollout
    # ────────────────────────────────────────────────────────────────────────
    
    def collect_rollout(self, env, state):
        """
        Run the environment for n_steps steps.
        Store (state, action, log_prob, reward, value, done) in the buffer.
        
        Returns the final state (to bootstrap the last value estimate)
        and the last done flag.
        """
        self.buffer.reset()
        
        ep_rewards     = []    # for tracking episode rewards during collection
        current_ep_r   = 0.0
        completed_ep_r = []    # rewards of episodes that finished in this rollout
        
        for _ in range(self.n_steps):
            action, log_prob, value = self.ac_net.get_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            self.buffer.add(state, action, log_prob, reward, value, done)
            
            current_ep_r += reward
            state = next_state
            
            if done:
                completed_ep_r.append(current_ep_r)
                current_ep_r = 0.0
                state, _ = env.reset()
        
        # Bootstrap: V(s_{T+1}) for computing advantages
        last_value = self.ac_net.get_value(state)
        self.buffer.compute_advantages(last_value, self.gamma, self.gae_lambda)
        
        return state, completed_ep_r
    
    # ────────────────────────────────────────────────────────────────────────
    # Phase 2: K-epoch update
    # ────────────────────────────────────────────────────────────────────────
    
    def update(self):
        """
        Run K epochs of gradient updates over the current rollout buffer.
        Each epoch shuffles the data into mini-batches.
        """
        ep_policy_loss = []; ep_value_loss = []
        ep_entropy = []; ep_clip_frac = []; ep_kl = []
        
        for _ in range(self.k_epochs):
            for states, actions, old_log_probs, advantages, returns in                     self.buffer.get_batches(self.mini_batch_size):
                
                total_loss, pl, vl, ent, cf = ppo_loss(
                    self.ac_net, states, actions, old_log_probs,
                    advantages, returns,
                    self.clip_epsilon, self.value_coef, self.entropy_coef
                )
                
                # Approximate KL divergence — early stopping signal
                # KL ≈ -mean(log_ratio) via log-ratio trick
                with torch.no_grad():
                    new_lp, _, _ = self.ac_net.evaluate_actions(states, actions)
                    log_ratio = new_lp - old_log_probs
                    approx_kl = (-log_ratio).mean().item()
                
                self.optimizer.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(self.ac_net.parameters(), self.max_grad_norm)
                self.optimizer.step()
                
                ep_policy_loss.append(pl)
                ep_value_loss.append(vl)
                ep_entropy.append(ent)
                ep_clip_frac.append(cf)
                ep_kl.append(approx_kl)
        
        # Record mean diagnostics for this update
        self.policy_losses.append(np.mean(ep_policy_loss))
        self.value_losses.append(np.mean(ep_value_loss))
        self.entropies.append(np.mean(ep_entropy))
        self.clip_fractions.append(np.mean(ep_clip_frac))
        self.approx_kls.append(np.mean(ep_kl))


# ── Sanity check ──────────────────────────────────────────────────────────────
agent_test = PPOAgent(n_states=8, n_actions=4)
print('PPO Agent initialized.')
print(f'  AC network params  : {sum(p.numel() for p in agent_test.ac_net.parameters()):,}')
print(f'  n_steps per rollout: {agent_test.n_steps}')
print(f'  k_epochs           : {agent_test.k_epochs}')
print(f'  clip epsilon (ε)   : {agent_test.clip_epsilon}')
print(f'  GAE lambda (λ)     : {agent_test.gae_lambda}')
print()
print('Phase 1 (collect): n_steps env steps → fill buffer → compute GAE')
print('Phase 2 (update) : k_epochs × (buffer_size / mini_batch) gradient steps')
print(f'  Updates per iteration: {agent_test.k_epochs} × ({agent_test.n_steps} / {agent_test.mini_batch_size}) = {agent_test.k_epochs * (agent_test.n_steps // agent_test.mini_batch_size)}')


## Cell 8 — Training Loop

PPO's training loop is **iteration-based**, not episode-based:
- Each iteration collects exactly `n_steps` environment steps
- Multiple complete episodes may occur within one rollout (or an episode may span multiple rollouts)
- After collecting, perform K epochs of gradient updates
- Track mean episode reward across completed episodes within each rollout

LunarLander typically takes 300–600 iterations (600K–1.2M steps) to solve.
We run 500 iterations here.


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
HP = dict(
    n_steps         = 1024,
    k_epochs        = 10,
    clip_epsilon    = 0.2,
    gamma           = 0.99,
    gae_lambda      = 0.95,
    lr              = 3e-4,
    value_coef      = 0.5,
    entropy_coef    = 0.01,
    mini_batch_size = 64,
    max_grad_norm   = 0.5,
)
N_ITERATIONS = 500   # × n_steps = 512K total env steps

# ── Initialize ────────────────────────────────────────────────────────────────
torch.manual_seed(SEED); np.random.seed(SEED)
env_train = gym.make('LunarLander-v3')
env_train.reset(seed=SEED)

agent = PPOAgent(n_states=8, n_actions=4, **HP)

# Tracking
all_ep_rewards  = []      # reward of every completed episode
iter_mean_r     = []      # mean ep reward per iteration (for plotting)
iter_mean_r100  = []      # rolling 100-ep mean
total_steps     = 0

# Initial state
state, _ = env_train.reset(seed=SEED)

print(f'Training PPO on LunarLander-v3')
print(f'Hyperparameters: {HP}')
print(f'Total env steps: {N_ITERATIONS * HP["n_steps"]:,}')
print()
print(f'{"Iter":>6}  {"Steps":>8}  {"EpisR":>8}  {"R100":>8}  '
      f'{"PLoss":>8}  {"VLoss":>8}  {"Entropy":>8}  {"ClipFr":>8}')
print('-' * 72)

t_start = time.time()
for iteration in range(1, N_ITERATIONS + 1):
    # ── Phase 1: Collect rollout ──────────────────────────────────────────────
    state, ep_rewards_this_iter = agent.collect_rollout(env_train, state)
    total_steps += HP['n_steps']
    
    if ep_rewards_this_iter:
        all_ep_rewards.extend(ep_rewards_this_iter)
        iter_mean_r.append(np.mean(ep_rewards_this_iter))
    else:
        iter_mean_r.append(iter_mean_r[-1] if iter_mean_r else 0.0)
    
    r100 = np.mean(all_ep_rewards[-100:]) if all_ep_rewards else 0.0
    iter_mean_r100.append(r100)
    
    # ── Phase 2: K-epoch update ───────────────────────────────────────────────
    agent.update()
    
    # ── Logging ───────────────────────────────────────────────────────────────
    if iteration % 50 == 0:
        pl = agent.policy_losses[-1]; vl = agent.value_losses[-1]
        ent = agent.entropies[-1];    cf = agent.clip_fractions[-1]
        elapsed = time.time() - t_start
        print(f'{iteration:>6d}  {total_steps:>8,}  {iter_mean_r[-1]:>8.1f}  '
              f'{r100:>8.1f}  {pl:>8.4f}  {vl:>8.2f}  {ent:>8.4f}  {cf:>8.4f}  '
              f'[{elapsed:.0f}s]')
    
    # Early stopping if solved
    if len(all_ep_rewards) >= 100 and r100 >= 200:
        print(f'\n✓ SOLVED at iteration {iteration} (step {total_steps:,})!')
        print(f'  Mean reward over last 100 episodes: {r100:.1f}')
        break

print(f'\nTraining finished. Total steps: {total_steps:,}')
env_train.close()


## Cell 9 — Training Curves and Diagnostics

In [ ]:
def smooth(data, w=10):
    if len(data) < w: return np.array(data)
    return np.convolve(data, np.ones(w)/w, mode='valid')

n_iter = len(iter_mean_r)
iters  = np.arange(1, n_iter + 1)

fig = plt.figure(figsize=(15, 11))
fig.suptitle('PPO Training on LunarLander-v3 — Full Diagnostics', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1. Episode reward + rolling 100
ax = fig.add_subplot(gs[0, :2])
ax.plot(iters, iter_mean_r,   alpha=0.3, color='steelblue', linewidth=1)
sm = smooth(iter_mean_r, 15)
ax.plot(np.arange(1, len(sm)+1), sm,           color='steelblue', linewidth=2.5, label='Smoothed (15-iter)')
ax.plot(iters, iter_mean_r100, color='orange',  linewidth=2,   label='Rolling 100-ep mean')
ax.axhline(200, color='red',  linestyle='--',  alpha=0.6,      label='Solved (200)')
ax.axhline(0,   color='gray', linestyle=':',   alpha=0.5)
ax.set_title('Episode Reward per Iteration'); ax.set_xlabel('Iteration'); ax.set_ylabel('Mean Reward')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 2. All episode rewards histogram
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(all_ep_rewards, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
ax2.axvline(200, color='red', linestyle='--', label='Solved (200)')
ax2.axvline(np.mean(all_ep_rewards), color='orange', linestyle='-',
            label=f'Mean ({np.mean(all_ep_rewards):.0f})')
ax2.set_title('Reward Distribution
(all episodes)'); ax2.set_xlabel('Reward')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# 3. Policy loss
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(agent.policy_losses, color='purple', linewidth=1.5)
ax3.set_title('Policy Loss (L^CLIP)'); ax3.set_xlabel('Update iteration')
ax3.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax3.grid(True, alpha=0.3)

# 4. Value loss
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(agent.value_losses, color='coral', linewidth=1.5)
ax4.set_title('Value Loss (L^VF)'); ax4.set_xlabel('Update iteration')
ax4.grid(True, alpha=0.3)

# 5. Entropy
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(agent.entropies, color='green', linewidth=1.5)
ax5.set_title('Policy Entropy
(decreases as policy sharpens)')
ax5.set_xlabel('Update iteration')
ax5.grid(True, alpha=0.3)

# 6. Clip fraction
ax6 = fig.add_subplot(gs[2, 0])
ax6.plot(agent.clip_fractions, color='brown', linewidth=1.5)
ax6.axhline(0.1, color='red', linestyle='--', alpha=0.5, label='Typical healthy range')
ax6.axhline(0.3, color='red', linestyle='--', alpha=0.5)
ax6.set_title('Clip Fraction
(fraction of samples clipped)')
ax6.set_xlabel('Update iteration'); ax6.legend(fontsize=8); ax6.grid(True, alpha=0.3)

# 7. Approx KL
ax7 = fig.add_subplot(gs[2, 1])
ax7.plot(agent.approx_kls, color='navy', linewidth=1.5)
ax7.axhline(0.02, color='orange', linestyle='--', alpha=0.7, label='KL~0.02 typical target')
ax7.set_title('Approximate KL Divergence
(policy change per update)')
ax7.set_xlabel('Update iteration'); ax7.legend(fontsize=8); ax7.grid(True, alpha=0.3)

# 8. All episode rewards over time
ax8 = fig.add_subplot(gs[2, 2])
if len(all_ep_rewards) > 0:
    ep_indices = np.arange(len(all_ep_rewards))
    ax8.scatter(ep_indices, all_ep_rewards, s=3, alpha=0.3, color='steelblue')
    sm_ep = smooth(all_ep_rewards, 20)
    ax8.plot(np.arange(len(sm_ep)), sm_ep, color='red', linewidth=2)
    ax8.axhline(200, color='green', linestyle='--', alpha=0.6)
ax8.set_title('All Episodes (scatter)'); ax8.set_xlabel('Episode #')
ax8.set_ylabel('Reward'); ax8.grid(True, alpha=0.3)

plt.show()


## Cell 10 — Policy Analysis: What Did the Agent Learn?

Inspect the trained policy's action preferences across key state dimensions.
A well-trained agent should:
- Fire main engine when y is low or y_vel is very negative (falling fast)
- Fire side engines to correct angle and x-position
- Do nothing (conserve fuel) when stable and well-positioned


In [ ]:
print('=== TRAINED POLICY INSPECTION ===')
print()

def get_probs_for_state(agent, state_arr):
    s_t = torch.tensor(state_arr, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, value = agent.ac_net(s_t)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs, value.item()

action_names = ['Nothing', 'L-Engine', 'Main-Eng', 'R-Engine']

test_states = {
    'Hovering perfectly (ideal)    ': [0.0,  0.5,  0.0,  0.0,  0.0,  0.0, 0, 0],
    'Falling fast (y_vel=-2)       ': [0.0,  0.5,  0.0, -2.0,  0.0,  0.0, 0, 0],
    'Tilted right (angle=+0.3)     ': [0.0,  0.5,  0.0,  0.0,  0.3,  0.0, 0, 0],
    'Tilted left  (angle=-0.3)     ': [0.0,  0.5,  0.0,  0.0, -0.3,  0.0, 0, 0],
    'Drifting right (x=0.5, vx=0.5)': [0.5,  0.5,  0.5,  0.0,  0.0,  0.0, 0, 0],
    'Very high up  (y=2.0)         ': [0.0,  2.0,  0.0, -0.1,  0.0,  0.0, 0, 0],
    'Near ground, slow (y=0.1)     ': [0.0,  0.1,  0.0, -0.2,  0.0,  0.0, 0, 0],
    'Legs touching                 ': [0.0,  0.05, 0.0, -0.1,  0.0,  0.0, 1, 1],
}

print(f'{"State":32s}  {"Nothing":>8}  {"L-Eng":>8}  {"Main":>8}  {"R-Eng":>8}  {"Best":>10}  {"V(s)":>8}')
print('-' * 95)
for desc, state in test_states.items():
    state_arr = np.array(state, dtype=np.float32)
    probs, val = get_probs_for_state(agent, state_arr)
    best_a = np.argmax(probs)
    print(f'{desc}  {probs[0]:>8.4f}  {probs[1]:>8.4f}  {probs[2]:>8.4f}  {probs[3]:>8.4f}  '
          f'{action_names[best_a]:>10}  {val:>8.2f}')

# Sweep y-velocity to see main engine firing preference
y_vels = np.linspace(-3.0, 0.5, 100)
probs_main = []
for yv in y_vels:
    state = np.array([0.0, 0.5, 0.0, yv, 0.0, 0.0, 0, 0], dtype=np.float32)
    probs, _ = get_probs_for_state(agent, state)
    probs_main.append(probs)
probs_main = np.array(probs_main)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Trained PPO Policy: Action Preferences', fontsize=12, fontweight='bold')

for i, (name, color) in enumerate(zip(action_names, ['gray','steelblue','red','orange'])):
    ax1.plot(y_vels, probs_main[:, i], color=color, linewidth=2, label=name)
ax1.axvline(0, color='black', linestyle=':', alpha=0.5)
ax1.set_xlabel('y velocity (negative = falling)'); ax1.set_ylabel('Action probability')
ax1.set_title('P(action) vs y velocity\n(x=0, y=0.5, other dims=0)')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Sweep angle
angles = np.linspace(-0.5, 0.5, 100)
probs_ang = []
for ang in angles:
    state = np.array([0.0, 0.5, 0.0, -0.5, ang, 0.0, 0, 0], dtype=np.float32)
    probs, _ = get_probs_for_state(agent, state)
    probs_ang.append(probs)
probs_ang = np.array(probs_ang)

for i, (name, color) in enumerate(zip(action_names, ['gray','steelblue','red','orange'])):
    ax2.plot(np.degrees(angles), probs_ang[:, i], color=color, linewidth=2, label=name)
ax2.axvline(0, color='black', linestyle=':', alpha=0.5, label='Upright')
ax2.set_xlabel('Pole angle (degrees)'); ax2.set_ylabel('Action probability')
ax2.set_title('P(action) vs lander angle\n(falling at -0.5 m/s)')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Cell 11 — Final Evaluation (30 Greedy Episodes)

In [ ]:
def evaluate_ppo(agent, n_episodes=30, verbose=True):
    env_eval = gym.make('LunarLander-v3')
    rewards  = []
    episode_lengths = []
    
    print(f'{"Ep":>4}  {"Reward":>9}  {"Steps":>7}  {"Result":>12}')
    print('-' * 42)
    
    for ep in range(n_episodes):
        state, _ = env_eval.reset()
        ep_reward = 0
        for step in range(1000):
            # Greedy: take action with highest probability
            action = agent.ac_net.get_action_greedy(state) if hasattr(agent.ac_net, 'get_action_greedy') else                      np.argmax(torch.softmax(agent.ac_net(torch.tensor(state,dtype=torch.float32).unsqueeze(0).to(device))[0],dim=-1).detach().cpu().numpy())
            state, reward, terminated, truncated, _ = env_eval.step(action)
            ep_reward += reward
            if terminated or truncated: break
        
        rewards.append(ep_reward)
        episode_lengths.append(step + 1)
        result = '✓ SUCCESS' if ep_reward >= 200 else ('~ OK' if ep_reward >= 0 else '✗ CRASH')
        if verbose:
            print(f'{ep+1:>4d}  {ep_reward:>9.1f}  {step+1:>7d}  {result:>12}')
    
    env_eval.close()
    print()
    print(f'Summary ({n_episodes} episodes):')
    print(f'  Mean reward     : {np.mean(rewards):.1f} ± {np.std(rewards):.1f}')
    print(f'  Min / Max       : {np.min(rewards):.1f} / {np.max(rewards):.1f}')
    print(f'  Episodes ≥ 200  : {sum(r>=200 for r in rewards)}/{n_episodes}')
    print(f'  Episodes ≥ 0    : {sum(r>=0   for r in rewards)}/{n_episodes}')
    return rewards, episode_lengths

# Add greedy method to actor-critic
def get_action_greedy(self, state):
    s_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = self.forward(s_t)
    return logits.argmax(dim=-1).item()
ActorCritic.get_action_greedy = get_action_greedy

print('=== FINAL EVALUATION ===')
eval_rewards, eval_lengths = evaluate_ppo(agent, n_episodes=30)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
colors_e = ['green' if r>=200 else ('orange' if r>=0 else 'red') for r in eval_rewards]
ax1.bar(range(1, len(eval_rewards)+1), eval_rewards, color=colors_e, edgecolor='white', alpha=0.85)
ax1.axhline(200, color='green',  linestyle='--', linewidth=1.5, label='Solved (200)')
ax1.axhline(0,   color='gray',   linestyle=':',  linewidth=1,   label='Zero reward')
ax1.axhline(np.mean(eval_rewards), color='blue', linewidth=1.5,
            label=f'Mean ({np.mean(eval_rewards):.0f})')
from matplotlib.patches import Patch
legend_p = [Patch(color='green',label='Success (≥200)'),
            Patch(color='orange',label='OK (0–199)'),
            Patch(color='red',  label='Crash (<0)')]
ax1.legend(handles=legend_p, fontsize=9)
ax1.set_title('Evaluation Rewards (30 episodes)')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward'); ax1.grid(True, alpha=0.3, axis='y')

ax2.hist(eval_rewards, bins=15, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(200, color='green', linestyle='--', label='Solved (200)')
ax2.axvline(np.mean(eval_rewards), color='blue', linestyle='-',
            label=f'Mean ({np.mean(eval_rewards):.0f})')
ax2.set_title('Reward Distribution'); ax2.set_xlabel('Reward'); ax2.set_ylabel('Count')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Cell 12 — Ablation: PPO With vs Without Clipping

This is the most important ablation. We train two agents:
- **PPO with clipping** (`clip_epsilon=0.2`) — standard PPO
- **Vanilla PG (no clipping)** (`clip_epsilon=∞`, effectively) — plain policy gradient with actor-critic

The no-clipping agent uses K epochs of updates on the same data without any constraint,
which often leads to instability, oscillation, or collapse.


In [ ]:
class PPOAgentNoClip(PPOAgent):
    """PPO without clipping — vanilla policy gradient with actor-critic."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    # Override update to skip clipping
    def update(self):
        for _ in range(self.k_epochs):
            for states, actions, old_log_probs, advantages, returns in                     self.buffer.get_batches(self.mini_batch_size):
                new_log_probs, values, entropy = self.ac_net.evaluate_actions(states, actions)
                # Standard policy gradient loss (no clipping, no ratio)
                policy_loss = -(new_log_probs * advantages).mean()
                value_loss  = F.mse_loss(values, returns)
                entropy_m   = entropy.mean()
                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy_m
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.ac_net.parameters(), self.max_grad_norm)
                self.optimizer.step()
                self.policy_losses.append(policy_loss.item())
                self.value_losses.append(value_loss.item())
                self.clip_fractions.append(0.0)


N_ITER_ABL = 200   # shorter run for comparison

def run_ppo_variant(use_clip, label, n_iter=N_ITER_ABL):
    torch.manual_seed(SEED); np.random.seed(SEED)
    env_a = gym.make('LunarLander-v3'); env_a.reset(seed=SEED)
    hp_abl = {**HP, 'n_steps': 512, 'k_epochs': 5}
    if use_clip:
        ag = PPOAgent(n_states=8, n_actions=4, **hp_abl)
    else:
        ag = PPOAgentNoClip(n_states=8, n_actions=4, **hp_abl)
    all_r = []; iter_r = []; state, _ = env_a.reset(seed=SEED)
    for i in range(n_iter):
        state, ep_rs = ag.collect_rollout(env_a, state)
        if ep_rs: all_r.extend(ep_rs); iter_r.append(np.mean(ep_rs))
        else: iter_r.append(iter_r[-1] if iter_r else 0.0)
        ag.update()
    env_a.close()
    final = np.mean(all_r[-50:]) if len(all_r) >= 50 else np.mean(all_r) if all_r else 0
    print(f'  {label:40s} | final mean (last 50 eps): {final:6.1f}')
    return iter_r, all_r

print('Training ablation agents (~3 min)...')
r_clip,   all_clip   = run_ppo_variant(True,  'PPO (clip_epsilon=0.2)')
r_noclip, all_noclip = run_ppo_variant(False, 'Vanilla PG (no clipping, K=5 epochs)')

def rolling_mean(data, w=20):
    return [np.mean(data[max(0,i-w):i+1]) for i in range(len(data))]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Ablation: PPO Clipping vs Vanilla Policy Gradient', fontsize=13, fontweight='bold')

iters_abl = np.arange(N_ITER_ABL)
axes[0].plot(iters_abl, rolling_mean(r_clip,   20), color='green', linewidth=2, label='PPO (with clip)')
axes[0].plot(iters_abl, rolling_mean(r_noclip, 20), color='red',   linewidth=2, label='No clip (vanilla PG)')
axes[0].axhline(200, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Rolling Mean Reward (20-iter)'); axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Mean Reward'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

rv_c  = [np.std(r_clip  [max(0,i-20):i+1]) for i in range(len(r_clip))]
rv_nc = [np.std(r_noclip[max(0,i-20):i+1]) for i in range(len(r_noclip))]
axes[1].plot(iters_abl, rv_c,  color='green', linewidth=2, label='PPO (with clip)')
axes[1].plot(iters_abl, rv_nc, color='red',   linewidth=2, label='No clip')
axes[1].set_title('Training Variance (rolling std)\n← clipping reduces instability')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Std Dev'); axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)

axes[2].hist(all_clip[-50:]   if len(all_clip)>=50   else all_clip,
             bins=20, alpha=0.65, color='green', label='PPO (with clip)')
axes[2].hist(all_noclip[-50:] if len(all_noclip)>=50 else all_noclip,
             bins=20, alpha=0.65, color='red',   label='No clip')
axes[2].axvline(200, color='gray', linestyle='--', alpha=0.5)
axes[2].set_title('Reward Distribution (last 50 eps)'); axes[2].set_xlabel('Reward')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


## Cell 13 — Visualizing the Clip Fraction Over Training

The **clip fraction** (fraction of samples where `|r_t - 1| > ε`) is a key health metric:
- **Too high (>0.3)**: policy is changing too fast — reduce learning rate or increase batch size
- **Too low (<0.02)**: policy is barely changing — possibly learning rate too small or converged
- **Healthy range: 0.05–0.20**


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('PPO Training Health Metrics (Full Training Run)', fontsize=13, fontweight='bold')

updates = np.arange(len(agent.policy_losses))

ax = axes[0,0]
ax.plot(updates, agent.clip_fractions, color='brown', linewidth=1.5, alpha=0.7)
sm_cf = smooth(agent.clip_fractions, 10)
ax.plot(np.arange(len(sm_cf)), sm_cf, color='brown', linewidth=2.5)
ax.axhspan(0.05, 0.20, alpha=0.1, color='green', label='Healthy range (0.05-0.20)')
ax.axhline(0.05, color='green', linestyle='--', alpha=0.5)
ax.axhline(0.20, color='green', linestyle='--', alpha=0.5)
ax.set_title('Clip Fraction'); ax.set_xlabel('Update step'); ax.set_ylabel('Fraction clipped')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[0,1]
ax.plot(updates, agent.approx_kls, color='navy', linewidth=1.5, alpha=0.7)
sm_kl = smooth(agent.approx_kls, 10)
ax.plot(np.arange(len(sm_kl)), sm_kl, color='navy', linewidth=2.5)
ax.axhspan(0.01, 0.03, alpha=0.1, color='blue', label='Target KL range (0.01-0.03)')
ax.set_title('Approximate KL Divergence\n(policy shift per update)')
ax.set_xlabel('Update step'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1,0]
ax.plot(updates, agent.entropies, color='teal', linewidth=1.5)
ax.set_title('Policy Entropy\n(decreases as policy becomes more decisive)')
ax.set_xlabel('Update step'); ax.set_ylabel('Entropy (nats)'); ax.grid(True, alpha=0.3)

ax = axes[1,1]
ax.plot(updates, agent.value_losses, color='coral', linewidth=1.5, alpha=0.7)
sm_vl = smooth(agent.value_losses, 10)
ax.plot(np.arange(len(sm_vl)), sm_vl, color='coral', linewidth=2.5)
ax.set_title('Value Function Loss\n(should decrease then stabilize)')
ax.set_xlabel('Update step'); ax.set_ylabel('MSE Loss'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


## Cell 14 — Summary and Next Steps

### What We Built From Scratch

| Component | Class/Function | Key Design Choice |
|-----------|---------------|-------------------|
| Actor-Critic network | `ActorCritic` | Shared backbone, two heads, orthogonal init |
| Advantage estimation | `compute_gae()` | Backward pass through TD residuals; λ controls bias/variance |
| Rollout buffer | `RolloutBuffer` | Fixed-size, collect-then-consume; mini-batch shuffling within K epochs |
| PPO clipped loss | `ppo_loss()` | `min(r·A, clip(r,1±ε)·A)` — pessimistic lower bound |
| Value loss | `ppo_loss()` | MSE between V(s) and return targets |
| Entropy bonus | `ppo_loss()` | Prevents premature policy collapse |
| Collect phase | `PPOAgent.collect_rollout()` | n_steps env steps; bootstrap last value |
| Update phase | `PPOAgent.update()` | K epochs × (n_steps/batch) gradient steps |

### PPO Health Checklist

| Metric | Healthy | Problem |
|--------|---------|---------|
| Clip fraction | 0.05–0.20 | >0.3: lr too high; <0.01: lr too low or converged |
| Approx KL | 0.01–0.03 | >0.05: consider adaptive KL penalty |
| Entropy | Gradually decreasing | Sudden drop: policy collapsed |
| Value loss | Decreasing | Not decreasing: critic not learning; may need higher lr_critic |

### The PPO Algorithm Family: Next Steps

- **PPO with adaptive KL**: instead of clipping, use a KL penalty that adapts its coefficient — original TRPO-style, slightly more principled
- **PPO-GAE (continuous)**: same algorithm with a **Gaussian policy** (output mean + std) instead of Categorical — used for robotics
- **MAPPO**: Multi-Agent PPO — each agent has its own policy, shared or independent critics
- **PPO + Curiosity**: Add an intrinsic reward based on prediction error for better exploration in sparse-reward environments

The full RL progression: Q-learning → DQN → REINFORCE → PPO (you are here) → SAC / TD3 (continuous, off-policy)
